In [0]:
orders_df=spark.read.format('csv').option("header","true").option("inferschema","true").load('/Volumes/ecommerce/retail/files/orders.csv')
orders_df.show()

+--------+-----------+----------+--------+-------------------+---------+
|order_id|customer_id|product_id|quantity|         order_date|   status|
+--------+-----------+----------+--------+-------------------+---------+
|       1|       4782|       387|       5|2024-07-04 00:32:04|  Pending|
|       2|       8077|       431|       3|2024-01-31 11:24:24|  Pending|
|       3|        100|        64|       3|2023-11-13 21:09:10|  Shipped|
|       4|       6877|       218|       4|2024-04-09 23:16:37|  Pending|
|       5|       5176|       250|       1|2024-04-03 16:53:38|  Pending|
|       6|       5654|       176|       4|2024-03-21 08:05:15|Cancelled|
|       7|       9193|       104|       1|2024-04-04 05:57:42|Cancelled|
|       8|       6440|       332|       4|2025-07-16 03:57:17|  Shipped|
|       9|       7570|       251|       4|2025-07-03 18:24:24|  Shipped|
|      10|       4819|       119|       3|2025-10-22 00:01:29|Delivered|
|      11|       6216|         3|       5|2024-04-0

In [0]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)



In [0]:
display(dbutils.fs.ls('/Volumes/ecommerce/retail/files'))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/retail/files/customers.csv,customers.csv,1140380,1781743482000
dbfs:/Volumes/ecommerce/retail/files/invoices.csv,invoices.csv,5602598,1781743482000
dbfs:/Volumes/ecommerce/retail/files/orders.csv,orders.csv,4656592,1781743482000
dbfs:/Volumes/ecommerce/retail/files/products.csv,products.csv,25348,1781743482000


In [0]:
%sql
list '/Volumes/ecommerce/retail/files'

path,name,size,modification_time
/Volumes/ecommerce/retail/files/customers.csv,customers.csv,1140380,1781743482000
/Volumes/ecommerce/retail/files/invoices.csv,invoices.csv,5602598,1781743482000
/Volumes/ecommerce/retail/files/orders.csv,orders.csv,4656592,1781743482000
/Volumes/ecommerce/retail/files/products.csv,products.csv,25348,1781743482000


In [0]:
#python  for data processing
#sql for datawarehousing
#volumes are buckets

In [0]:
%sql
show catalogs

catalog
ecommerce
samples
system
workspace


In [0]:
%sql
show volumes in ecommerce.retail

database,volume_name
retail,files


In [0]:
single_df=spark.read.format('json').load('/Volumes/ecommerce/retail/files/singleline.json')
singleline_df.show()

+--------------------+---+--------------------+--------+
|             address|age|             hobbies|    name|
+--------------------+---+--------------------+--------+
|{New York, 123 Ma...| 30|[reading, traveli...|John Doe|
+--------------------+---+--------------------+--------+



In [0]:
display(singleline_df)


address,age,hobbies,name
"List(New York, 123 Main St)",30,"List(reading, traveling, swimming)",John Doe


In [0]:
multiline_df=spark.read.format('json').option('multiline','true').load('/Volumes/ecommerce/retail/files/multiline.json')
multiline_df.show()


+--------------------+---+--------------------+--------+
|             address|age|             hobbies|    name|
+--------------------+---+--------------------+--------+
|{New York, 123 Ma...| 30|[reading, traveli...|John Doe|
+--------------------+---+--------------------+--------+



In [0]:
display(multiline_df)

address,age,hobbies,name
"List(New York, 123 Main St)",30,"List(reading, traveling, swimming)",John Doe


In [0]:
multiline_df.count()

1

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField(
        "address", 
        StructType([
            StructField("street", StringType(), True),
            StructField("city", StringType(), True)
        ]), 
        True
    ),
    StructField("hobbies", StringType(), True)
])

In [0]:
display(schema)

StructType([StructField('name', StringType(), True), StructField('age', IntegerType(), True), StructField('address', StructType([StructField('street', StringType(), True), StructField('city', StringType(), True)]), True), StructField('hobbies', StringType(), True)])

In [0]:
test_schema_df=spark.read.format('json').option('multiline','true').schema(schema).load('/Volumes/ecommerce/retail/files/multiline.json')

In [0]:
test_schema_df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- city: string (nullable = true)
 |-- hobbies: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
display(test_schema_df)

name,age,address,hobbies
John Doe,30,"List(123 Main St, New York)","List(reading, traveling, swimming)"


In [0]:
display(multiline_df)

address,age,hobbies,name
"List(New York, 123 Main St)",30,"List(reading, traveling, swimming)",John Doe


In [0]:
multiline_df.select('name').show()

+--------+
|    name|
+--------+
|John Doe|
+--------+



In [0]:
multiline_df.select('name','address.street').show()

+--------+-----------+
|    name|     street|
+--------+-----------+
|John Doe|123 Main St|
+--------+-----------+



In [0]:
from pyspark.sql.functions import explode
multiline_df.select('name','address.street',explode('hobbies').alias ('hobbies')).show()

+--------+-----------+---------+
|    name|     street|  hobbies|
+--------+-----------+---------+
|John Doe|123 Main St|  reading|
|John Doe|123 Main St|traveling|
|John Doe|123 Main St| swimming|
+--------+-----------+---------+



In [0]:
test_schema_df.select('name','address.street',explode('hobbies')).show()

+--------+-----------+---------+
|    name|     street|      col|
+--------+-----------+---------+
|John Doe|123 Main St|  reading|
|John Doe|123 Main St|traveling|
|John Doe|123 Main St| swimming|
+--------+-----------+---------+



In [0]:
read_null_df=spark.read.format('json').load('/Volumes/ecommerce/retail/files/reading_null.json')

In [0]:
read_null_df.printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- street: string (nullable = true)
 |-- age: long (nullable = true)
 |-- hobbies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- name: string (nullable = true)



In [0]:
read_null_df.show()

+--------------------+----+--------------------+--------+
|             address| age|             hobbies|    name|
+--------------------+----+--------------------+--------+
|{New York, 123 Ma...|  30|[reading, traveli...|John Doe|
|{New York, 123 Ma...|  30|[reading, traveli...|      Jo|
| {NULL, 123 Main St}|NULL|[reading, traveli...|     Doe|
|{New York, 123 Ma...|  30|                NULL|    John|
+--------------------+----+--------------------+--------+



In [0]:
display(read_null_df)

address,age,hobbies,name
"List(New York, 123 Main St)",30,"List(reading, traveling, swimming)",John Doe
"List(New York, 123 Main St)",30,"List(reading, traveling, swimming)",Jo
"List(null, 123 Main St)",null,"List(reading, traveling, swimming)",Doe
"List(New York, 123 Main St)",30,null,John


In [0]:
read_null_df.fillna({'age':0}).show()

+--------------------+---+--------------------+--------+
|             address|age|             hobbies|    name|
+--------------------+---+--------------------+--------+
|{New York, 123 Ma...| 30|[reading, traveli...|John Doe|
|{New York, 123 Ma...| 30|[reading, traveli...|      Jo|
| {NULL, 123 Main St}|  0|[reading, traveli...|     Doe|
|{New York, 123 Ma...| 30|                NULL|    John|
+--------------------+---+--------------------+--------+



In [0]:
read_null_df.filter(read_null_df.age.isNotNull()).show()

+--------------------+---+--------------------+--------+
|             address|age|             hobbies|    name|
+--------------------+---+--------------------+--------+
|{New York, 123 Ma...| 30|[reading, traveli...|John Doe|
|{New York, 123 Ma...| 30|[reading, traveli...|      Jo|
|{New York, 123 Ma...| 30|                NULL|    John|
+--------------------+---+--------------------+--------+



In [0]:
read_null_df.filter(read_null_df.age.isNull()).show()

+-------------------+----+--------------------+----+
|            address| age|             hobbies|name|
+-------------------+----+--------------------+----+
|{NULL, 123 Main St}|NULL|[reading, traveli...| Doe|
+-------------------+----+--------------------+----+



In [0]:
read_null_df.select('name',explode('hobbies').alias('hobbies')).show()
#explode drops null records

+--------+---------+
|    name|  hobbies|
+--------+---------+
|John Doe|  reading|
|John Doe|traveling|
|John Doe| swimming|
|      Jo|  reading|
|      Jo|traveling|
|      Jo| swimming|
|     Doe|  reading|
|     Doe|traveling|
|     Doe| swimming|
+--------+---------+



In [0]:
from pyspark.sql.functions import explode_outer
read_null_df.select('name',explode_outer('hobbies').alias('hobbies')).show()
#explode_outer keeps null records

+--------+---------+
|    name|  hobbies|
+--------+---------+
|John Doe|  reading|
|John Doe|traveling|
|John Doe| swimming|
|      Jo|  reading|
|      Jo|traveling|
|      Jo| swimming|
|     Doe|  reading|
|     Doe|traveling|
|     Doe| swimming|
|    John|     NULL|
+--------+---------+



In [0]:
read_null_df.select('name','address.city').show()

+--------+--------+
|    name|    city|
+--------+--------+
|John Doe|New York|
|      Jo|New York|
|     Doe|    NULL|
|    John|New York|
+--------+--------+



In [0]:
new_read_null_df=read_null_df.fillna({'address.city':'New York'}).show()

+--------------------+----+--------------------+--------+
|             address| age|             hobbies|    name|
+--------------------+----+--------------------+--------+
|{New York, 123 Ma...|  30|[reading, traveli...|John Doe|
|{New York, 123 Ma...|  30|[reading, traveli...|      Jo|
| {NULL, 123 Main St}|NULL|[reading, traveli...|     Doe|
|{New York, 123 Ma...|  30|                NULL|    John|
+--------------------+----+--------------------+--------+



In [0]:
from pyspark.sql.functions import col, coalesce, lit
read_null_df=read_null_df.withColumn('address',col('address').withField('city',coalesce(col('address.city'),lit('unknown'))))
read_null_df.show()

#new_read_null_df=read_null_df.withColumn('address',col('address').withField('city',coalesce(col('address.city'),lit('unknown'))))

+--------------------+----+--------------------+--------+
|             address| age|             hobbies|    name|
+--------------------+----+--------------------+--------+
|{New York, 123 Ma...|  30|[reading, traveli...|John Doe|
|{New York, 123 Ma...|  30|[reading, traveli...|      Jo|
|{unknown, 123 Mai...|NULL|[reading, traveli...|     Doe|
|{New York, 123 Ma...|  30|                NULL|    John|
+--------------------+----+--------------------+--------+

